In [5]:
import pandas as pd
import numpy as np
import os
import re

In [6]:
event_files=[os.path.join(root,f) for root,dirs,files in os.walk('../bids') for f in files if f.endswith('events.tsv')]
data=[]
for file in event_files:
    sub='sub-'+re.search('func/sub-(.*)_task',file).group(1)
    acq=re.search('_acq-(.*)_events',file).group(1)
    tmp_df=pd.read_csv(file,sep='\t')
    if tmp_df.shape[0]>0:
        print(sub,acq)
        tmp_df['sub']=sub
        tmp_df['acq']=acq
        data.append(tmp_df)
events_df=pd.concat(data)


sub-10041 mb6me4
sub-10041 mb3me1
sub-10041 mb1me1
sub-10041 mb1me4
sub-10041 mb3me4
sub-10041 mb6me1
sub-10085 mb1me1
sub-10085 mb6me4
sub-10085 mb6me1
sub-10085 mb3me1
sub-10085 mb1me4
sub-10085 mb3me4
sub-10035 mb1me1
sub-10035 mb6me1
sub-10035 mb3me4
sub-10035 mb6me4
sub-10035 mb3me1
sub-10035 mb1me4
sub-10074 mb6me1
sub-10074 mb3me1
sub-10074 mb6me4
sub-10074 mb3me4
sub-10074 mb1me4
sub-10074 mb1me1
sub-10043 mb3me1
sub-10043 mb1me1
sub-10043 mb1me4
sub-10043 mb6me1
sub-10043 mb3me4
sub-10043 mb6me4
sub-10069 mb6me1
sub-10069 mb3me4
sub-10069 mb3me1
sub-10069 mb1me4
sub-10069 mb1me1
sub-10069 mb6me4
sub-10059 mb1me4
sub-10059 mb1me1
sub-10059 mb6me4
sub-10059 mb3me1
sub-10059 mb3me4
sub-10059 mb6me1
sub-10017 mb6me4
sub-10017 mb1me4
sub-10017 mb3me1
sub-10017 mb1me1
sub-10017 mb3me4
sub-10017 mb6me1
sub-10080 mb1me4
sub-10080 mb1me1
sub-10080 mb6me4
sub-10080 mb3me4
sub-10080 mb6me1
sub-10080 mb3me1
sub-10094 mb1me4
sub-10094 mb6me1
sub-10094 mb6me4
sub-10094 mb3me4
sub-10094 mb1m

In [7]:
print(events_df['sub'].unique())

['sub-10041' 'sub-10085' 'sub-10035' 'sub-10074' 'sub-10043' 'sub-10069'
 'sub-10059' 'sub-10017' 'sub-10080' 'sub-10094' 'sub-10054' 'sub-10024']


In [8]:
data=[]
for sub in events_df['sub'].unique():
    print(sub)
    for acq in events_df['acq'].unique():
        
        absolute=np.loadtxt('../derivatives/fsl/mcflirt/%s/%s/_abs.rms'%(sub,acq))
        FD=np.loadtxt('../derivatives/fsl/mcflirt/%s/%s/_rel.rms'%(sub,acq))
        
        row=[sub,acq,
             np.divide(
                 events_df[(events_df['sub']==sub)&(events_df['acq']==acq)]['trial_type'].str.count('miss').sum()
                 ,2),
            absolute.max(),FD.mean()]
        data.append(row)
        
exclusions_df=pd.DataFrame(data=data,columns=['sub','acq','TrialCount_misses','Max_Abs_motion','FD_mean'])
exclusions_df['FD_exclusion']=exclusions_df['FD_mean']>0.5
exclusions_df['ABS_exclusion']=exclusions_df['Max_Abs_motion']>1.35
exclusions_df['Beh_TrialExclusion']=exclusions_df['TrialCount_misses']>27


sub-10041
sub-10085
sub-10035
sub-10074
sub-10043
sub-10069
sub-10059
sub-10017
sub-10080
sub-10094
sub-10054
sub-10024


In [9]:

results=exclusions_df.groupby(by='sub').sum().reset_index().rename(columns={"TrialCount_misses": "TotalCount_misses"})
results['Beh_TotalExclusion']=results['TotalCount_misses']>81
results=results[['sub','TotalCount_misses','Beh_TotalExclusion']]


In [10]:
exclusions_df.merge(results,on='sub')
exclusions_df.to_csv('../derivatives/exclusions.csv', index=False)

In [14]:
exclusions_df.groupby(by='sub').sum()

,TrialCount_misses,Max_Abs_motion,FD_mean,FD_exclusion,ABS_exclusion,Beh_TrialExclusion
sub,,,,,,
sub-10017,1.0,4.135335,0.280834,0,0,0
sub-10024,4.0,3.286222,0.333501,0,0,0
sub-10035,33.0,2.748913,0.300442,0,0,0
sub-10041,0.0,8.948181,0.595730,0,4,0
sub-10043,3.0,4.666819,0.334741,0,0,0
sub-10054,13.0,4.730377,0.499728,0,1,0
sub-10059,0.0,3.283416,0.289011,0,0,0
sub-10069,7.0,7.064291,0.586771,0,2,0
sub-10074,4.0,4.583511,0.365633,0,0,0
